In [1]:
# 设置：切到项目根目录，确保 %run minigpt/... 与相对路径可用（无论从哪个目录启动 Jupyter）
import os
from pathlib import Path

_root = Path.cwd()
while not (_root / "minigpt").exists() and _root != _root.parent:
    _root = _root.parent
os.chdir(_root)


## 1.引言

通过前面两篇文章[带你实现多头注意力](https://golfxiao.blog.csdn.net/article/details/143697790)和[带你构建TransformerBlock](https://golfxiao.blog.csdn.net/article/details/143741939)，我们基本已经构建完了一个大语言模型的关键模块，本节我们将基于这些模块构建出一个能够运行的GPT Model，并用这个Model类完成从序列文本输入到序列文本输出的整个流程。整个流程如下所示：

![GPT](../img/9-4.png)

- **序列化**：用户输入的文本首先通过分词器处理，转化为一系列的token ID；
- **嵌入**：离散的token ID序列经过嵌入层（embedding layer）处理后，为每个token生成对应的连续向量表示； 
- **推理**：将这些序列向量输入模型，经过一系列连续的矩阵乘法运算，模型将基于已有的上下文信息预测出下一个token的概率分布； 
- **选词**：根据概率分布，模型选取下一个最可能的token，从而生成连贯的输出token序列。
- **反序列化**：通过分词器的反序列化，将选出的token ID转换回可读的文本形式。

## 2.模型架构组成

我们将要构建的LLM架构组成如下：

![GPT架构](../img/9-1.jpg)

- **嵌入层（embedding layer）**: 用于将tokenID序列转换为连续向量表示； 
- **丢弃层（embedding dropout）**: 用于在训练过程中随机丢弃部分嵌入向量，减少模型的过拟合现象； 
- **解码层（decode layers）**: 由多个TransformerBlock堆叠而成，负责处理输入并逐步得到序列的上下文向量。
- **最终归一化（final norm）**: 对所有输出进行最一化处理，确保输出数值在一个适当的范围内； 
- **输出层（output layer）**: 将模型的最终向量转换为token的概率分布； 

下面，我们将一步一步创建出每一层，并用一个输入数据演示每一层运算后的输出效果。

#### 2.1 准备工作

首先，我们引入前面已经构建的transformer block组件，其它组件包括多头自注意力也已经包含在此脚本中。

In [2]:
%run minigpt/model/transformer.py

定义一套配置项，用于设置模型的基本结构，包括：
- vocab_size: 词表大小, 这里使用分词器训练的词表大小32000; 
- emb_dim: 嵌入层维度，设置为4是为了演示方便；
- n_heads: 多头数量，emb_dim必须是n_heads的整数倍，这里设置为2； 
- context_length: 上下文长度, 为了演示方便，设置为10; 
- n_layers: 解码层数； 
- drop_rate: 训练中dropout层随机丢弃的参数比例； 

In [3]:
config = {
    "vocab_size": 32000, # Vocabulary size
    "context_length": 10, # Context length
    "emb_dim": 4, # Embedding dimension
    "n_heads": 2, # Number of attention heads
    "n_layers": 2, # Number of layers
    "drop_rate": 0.1, # Dropout rate
    "qkv_bias": False, # Query-Key-Value bias
}

用随机数创建一个batch_size=2, seq_len=4的小批量输入，以这个数据为例来演示模型的前向传播过程。

In [4]:
inputs = torch.randint(low=0, high=config['vocab_size'], size=(2, 4))
b, seq_len = inputs.shape
print(f"batch_size:{b}, seq_len: {seq_len}")
print("inputs:", inputs)

batch_size:2, seq_len: 4
inputs: tensor([[25709, 16016, 27838, 17917],
        [ 8122,  2946, 30100, 26739]])


#### 创建过程

**第一步**：创建嵌入层。

根据配置创建一个词表大小为32000、嵌入维度为4的嵌入层，并对小批量输入序列进行向量嵌入。

In [5]:
token_emb = nn.Embedding(config['vocab_size'], config['emb_dim'])
x = token_emb(inputs)
x

tensor([[[ 0.3359,  0.0259,  1.5485,  0.3580],
         [ 0.6186,  1.5642, -2.1290, -0.2225],
         [-0.3804,  0.1623, -0.7665, -1.0547],
         [-0.6624, -0.4568,  0.1877, -0.4593]],

        [[-1.0212,  0.2722, -0.6244, -0.9958],
         [ 0.3784, -0.1594, -0.3836,  0.3331],
         [-1.0129,  2.3594,  0.5360,  0.7094],
         [ 0.9115,  0.2977, -0.2709, -0.2117]]], grad_fn=<EmbeddingBackward0>)

**第二步**：创建一个丢弃层，drop_rate=0.1表示每次训练时约有10%的嵌入会被丢弃。
疑问：一个[2,4,4]的数据，为何32个参数只丢弃了1个元素

In [6]:
drop_emb = nn.Dropout(config['drop_rate'])
x = drop_emb(x)
x

tensor([[[ 0.3732,  0.0288,  1.7206,  0.3978],
         [ 0.6874,  1.7380, -2.3656, -0.2472],
         [-0.4226,  0.1803, -0.8517, -1.1719],
         [-0.0000, -0.5075,  0.2086, -0.5104]],

        [[-1.1346,  0.3024, -0.6938, -1.1065],
         [ 0.4204, -0.1771, -0.4262,  0.3701],
         [-0.0000,  2.6215,  0.5956,  0.7882],
         [ 1.0128,  0.0000, -0.3010, -0.2353]]], grad_fn=<MulBackward0>)

**第三步**：计算位置编码

由于解码层中的注意力得分计算需要使用位置编码，这里先根据`context_length`计算出0-9每个位置的旋转编码，然后截取此次输入长度（seq_len=4)范围内的部分作为输入序列inputs的位置编码。

In [7]:
pos_cis = precompute_pos_cis(config['emb_dim'] // config['n_heads'], config['context_length'])
pos_cis = pos_cis[:seq_len]
pos_cis

tensor([[ 1.0000+0.0000j],
        [ 0.5403+0.8415j],
        [-0.4161+0.9093j],
        [-0.9900+0.1411j]])

**第四步**：创建解码层

根据n_layers参数循环构建出指定数量的解码层，并使用此解码层序列对输入向量作特征计算，最终得到一个能表示输入序列特征的上下文向量。

> 注：每一个解码层是[前文](https://golfxiao.blog.csdn.net/article/details/143741939)讲过的TransformerBlock块实例，里面封装了层归一化、多头注意力、前馈神经网络、残差连接等transformer核心组件。

In [8]:
decode_layers = nn.Sequential(*[
    TransformerBlock(**config) for _ in range(config['n_layers'])
])
for i, block in enumerate(decode_layers):
    x, _ = block(x, pos_cis)
print(x)

tensor([[[-0.0654,  2.3860,  2.1787,  2.2806],
         [ 0.8731,  2.2183, -2.6357, -0.0176],
         [-0.7417,  1.2746, -1.2560, -1.1901],
         [ 0.6327, -0.1253,  0.5373,  0.0647]],

        [[-0.9965,  0.3931, -1.1579, -1.0530],
         [ 1.5744, -0.3978, -0.9558,  0.0193],
         [-0.1832,  2.5977,  0.1023,  0.9132],
         [ 1.6940,  0.1016, -0.9865, -0.7771]]], grad_fn=<AddBackward0>)



**第五步**：创建最终归一化层，并对解码层计算出的上下文向量进行归一化操作。

In [9]:
final_norm = LayerNorm(config['emb_dim'])
x = final_norm(x)
x

tensor([[[-1.4961,  0.5873,  0.4111,  0.4977],
         [ 0.3728,  1.0297, -1.3405, -0.0621],
         [-0.2212,  1.4721, -0.6531, -0.5978],
         [ 0.9717, -1.1011,  0.7108, -0.5814]],

        [[-0.3989,  1.4937, -0.6188, -0.4760],
         [ 1.3948, -0.4217, -0.9356, -0.0375],
         [-0.8328,  1.3926, -0.6043,  0.0446],
         [ 1.3833,  0.0768, -0.8159, -0.6442]]], grad_fn=<AddBackward0>)

> 可以看到，经过归一化后，每个张量的数值范围明显收窄。

**第六步**：创建线性层（Linear Layer），用于将最后的特征映射到词汇表的大小，生成每个token的预测概率。

In [10]:
out_head = nn.Linear(config['emb_dim'], config['vocab_size'])
logits = out_head(x)
logits, logits.shape

(tensor([[[-0.1566,  0.2250,  0.3171,  ..., -0.0543, -0.1241, -1.4066],
          [-1.1945,  0.7661, -0.9486,  ..., -0.8505,  0.5809,  0.2456],
          [-0.5872,  0.6725, -0.8700,  ..., -0.9317, -0.1025, -0.3099],
          [ 0.0151, -0.6496, -0.8038,  ...,  0.4549,  0.6523,  0.3536]],
 
         [[-0.5814,  0.6899, -0.7516,  ..., -0.9046, -0.1284, -0.4450],
          [-1.1037,  0.1180, -1.0976,  ..., -0.1713,  1.2151,  0.9360],
          [-0.6695,  0.7258, -0.3692,  ..., -0.7531, -0.0488, -0.7607],
          [-0.8617,  0.1835, -1.3673,  ..., -0.4540,  0.8180,  0.9004]]],
        grad_fn=<ViewBackward0>),
 torch.Size([2, 4, 32000]))

> 输出的logtis是一个（2，4，32000）形状的张量，表示每个位置的next token预测结果是一个32000维度的向量，即词表中每个token的可能性分数。

## 3.模型封装

#### 3.1 模型配置封装
前面用字典形式创建的模型配置虽然方便，但不够规范，由于兼容transformer库已经成为开源模型的一个事实标准，所以我们也采用结构化的方式来定义模型配置。具体如下 ：
- 使用`transformers.PretrainedConfig`作为模型配置的基类（transformers库的标准）。
- 所有配置参数使用类的成员属性显式定义，这样可以提供默认值，并限制类型。
- 定义一个类属性字段model_type，作为模型独一无二的类型标识，类似通义千问的`qwen`一样，用以和transformers库中其它类型的模型区分。

In [11]:
class GPTConfig(PretrainedConfig):
    # 每个模型都必须有一个独特的model_type，否则会报"Should have a `model_type` key in its config.json"
    model_type = "minigpt"

    def __init__(self, **kwargs):
        self.context_length = kwargs.get('context_length', 1024)
        self.vocab_size = kwargs.get('vocab_size', 32000)
        self.emb_dim = kwargs.get('emb_dim', 768)
        self.drop_rate = kwargs.get('drop_rate', 0.1)
        self.n_layers = kwargs.get('n_layers', 12)
        self.n_heads = kwargs.get('n_heads', 12)
        self.qkv_bias = kwargs.get('qkv_bias', False)
        super().__init__(**kwargs)

cfg = GPTConfig()
cfg

GPTConfig {
  "context_length": 1024,
  "drop_rate": 0.1,
  "emb_dim": 768,
  "model_type": "minigpt",
  "n_heads": 12,
  "n_layers": 12,
  "qkv_bias": false,
  "transformers_version": "5.1.0",
  "vocab_size": 32000
}

> 上面的各个配置项的默认值，就是我们将要构建的模型目标结构，这里采用768维的向量嵌入，12个解码层，12个注意力头，1024的上下文长度。

#### 3.2 模型结构封装

我们最终创建一个名为MiniGPT的模型类，为了与HuggingFace的transformers库兼容，该类需要满足两点：
1. 继承自PreTrainedModel
2. 采用PreTrainedConfig类型的配置对象作为构造函数参数。

> 标准化的基类还提供了以下好处：
> - 可以直接使用from_pretrained和save_pretrained方法来加载模型权重和保存模型权重，不用关心权重存储细节； 
> - 可以集中管理与模型结构相关的超参数，使得模型的构建代码非常简洁一致，与便于配置的扩展。

具体操作就是将上面嵌入层、dropout层、解码层、最终归一化层、输出层的创建代码封装到构造方法中。

In [12]:
class MiniGPT(PreTrainedModel):
    config_class = GPTConfig

    def __init__(self, config: GPTConfig):
        super().__init__(config)
        self.context_length = config.context_length
        self.num_heads = config.n_heads
        self.n_layers = config.n_layers
        self.token_emb = nn.Embedding(config.vocab_size, config.emb_dim)
        self.drop_emb = nn.Dropout(config.drop_rate)
        
        pos_cis = precompute_pos_cis(config.emb_dim // config.n_heads, config.context_length)
        self.register_buffer("pos_cis", pos_cis, persistent=False)
        self.decode_layers = nn.Sequential(*[
            TransformerBlock(**(config.to_dict())) for _ in range(config.n_layers)
        ])
        
        self.final_norm = LayerNorm(config.emb_dim)
        self.out_head = nn.Linear(config.emb_dim, config.vocab_size)

#### 3.3 模型推理封装
将前面各个层的运算代码封装到forward方法中。

In [13]:
def forward(self, inputs:torch.Tensor, **kwargs):
    b, seq_len = inputs.shape
    pos_cis = self.pos_cis[:seq_len]
    x = self.token_emb(inputs)
    x = self.drop_emb(x)

    for i, block in enumerate(self.decode_layers):
        x, _ = block(x, pos_cis)

    x = self.final_norm(x)
    logits = self.out_head(x)
    return logits

setattr(MiniGPT, "forward", forward)


> Tips1：定义模型对外的forward时，最好预留一个kwargs参数，用于接受一些训练器额外传递的参数，例如注意力掩码、kv_cache开关等，如果没有kwargs，python方法接收未声明的参数时会报错。

> Tips2：很多文章都提到dropout只会用在训练模式下，但上面的drop_emb方法我们并不需要显式加模式判断，原因在于Dropout组件内部实现了train()和eval()模式的判断，只有在train模式下才会进行随机丢弃。有相同行为的组件还有BatchNorm，只有train模式下才会使用批次统计信息，推理模式下会使用整个训练数据的统计信息。

#### 3.4 模型测试
下面我们初始化这个模型，以随机初始化的批次序列来查看模型的输出。

In [14]:
torch.manual_seed(123)
batch = torch.randint(low=0, high=cfg.vocab_size, size=(2, 4))
model = MiniGPT(cfg)
logits = model(batch)
print("inputs: ", batch)
print("output shape:", logits.shape)
print("outputs:", logits)

inputs:  tensor([[16382,  7789, 31102, 12610],
        [15580, 31842,  6886, 23057]])
output shape: torch.Size([2, 4, 32000])
outputs: tensor([[[ 0.7860, -0.2217, -0.1942,  ...,  0.9404,  0.2453, -0.2208],
         [ 0.5764,  0.2916,  0.5871,  ...,  0.8125,  0.2281,  0.1774],
         [-0.7044,  0.0051, -0.3743,  ..., -0.2570,  0.9257, -0.1143],
         [ 0.2783,  0.1833,  0.0912,  ...,  0.3704,  0.0329,  0.0647]],

        [[ 0.6852, -0.3530, -0.5964,  ..., -0.7108,  0.5489,  0.3886],
         [ 0.4020, -0.1560,  0.1957,  ...,  0.4832,  0.4865, -0.4783],
         [ 0.1024,  0.2261, -0.4227,  ..., -0.4512, -0.1861, -0.2825],
         [ 0.8843,  0.3839,  0.7342,  ...,  0.3111, -0.2077, -0.0667]]],
       grad_fn=<ViewBackward0>)


通过`model.parameters()`可以计算模型的参数量。

In [15]:
total_params = sum(param.numel() for param in model.parameters())
total_params

134212352

此模型有1.34亿个参数。以32位浮点数精度来计算模型参数的内存需求：

In [16]:
total_size_bytes = total_params * 4
print(f"total_size of the model: {total_size_bytes/(1024*1024):.2f}MB")

total_size of the model: 511.98MB


## 4.生成文本序列

我们对模型的需求是输出一个完整的文本序列，而上面构建的MiniGPT仅仅是输出下一个token的logits。为了生成一个序列我们需要对GPTModel进行多次迭代调用，每次迭代得到一个token，再把这个token添加到序列中继续迭代，整个过程类似下图所示。
![LLM模型](../img/9-2.jpg)

为此，需要编写一个generate函数来完成一个序列的预测。

In [17]:
@torch.inference_mode
def generate(self, input_ids, max_length=512, eos_token_id=-1):
    # 创建batch长度的全零值，用作停止推理的判断。
    eos_reached = torch.zeros(len(input_ids), dtype=torch.bool, device=input_ids.device)
    for _ in range(max_length):
        # 如果生成序列过程中超出上下文长度，则由后往前截取context_length个token。
        context_ids = input_ids[:, -self.context_length:]  
        with torch.no_grad():
            output = self(context_ids)  # shape: batch, n_tokens, vocab_size

        # 只取每个序列最后一个token的输出向量作为logits, shape变为: batch, vocab_size
        logits = output[:, -1, :]        
        # 使用softmax函数将logits转换为下一个token的概率分布，shape仍是: batch, vocab_size
        probs = torch.softmax(logits, dim=-1)   
        # 取概率最大的作为next_token_ids，形状变为：batch, 1
        next_token_ids = torch.argmax(probs, dim=-1, keepdim=True)
        # 将next_token_id连接到下一个token的结尾， 形状变为：batch, n_tokens+1
        input_ids = torch.cat((input_ids, next_token_ids), dim=1)
        
        # 更新 eos_reached，需要所有batch都推理出eos_token，才会终止推理
        eos_reached |= (next_token_ids.squeeze(-1) == eos_token_id)
        if eos_reached.all(): break

    return input_ids

setattr(MiniGPT, "generate", generate)

上面函数中for循环内部的代码逻辑，是在完成一个token的预测，代码逻辑可以按照下图所示的流程来辅助理解：
![next token预测](../img/9-3.jpg)

在测试这个generate函数之前，需要先创建一个Tokenizer类，用于文本到序列的转换，这里使用之前训练好的分词器`tokenizer_v3`。

In [18]:
from transformers import AutoTokenizer

tokenizer_path = "models/tokenizer_v3"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)

input_text = "Hello, I am"
input_ids = tokenizer.encode(input_text)
input_ids

[4313, 14, 1923, 5570]

接下来，就用这个输入序列来测试generate函数。

In [19]:
batch = torch.tensor(input_ids).unsqueeze(0)
eos_token = tokenizer.eos_token_id
generated_seqs = model.generate(batch, 5, tokenizer.eos_token_id)
print("inputs: ", batch)
print("output:", generated_seqs)

inputs:  tensor([[4313,   14, 1923, 5570]])
output: tensor([[ 4313,    14,  1923,  5570, 16948,  5106, 24499,  4254,  2295]])


> unsqueeze方法用于扩展维度，上面的代码是按照指定的维度`0`将张量由形状为[6]->[1, 6]，扩展维度的原因是模型只接受批量输入，这里即使只有一个序列，也需要将张量扩展成批次输入的形状。

下面使用decode方法将输出的数字序列转换为文本。

In [20]:
tokenizer.decode(generated_seqs.squeeze(0))

'Hello, I am占比领域的尤为29ol'

> squeeze方法用于压缩维度，上面的示例中是将第0维去掉，张量形状由[1, 11]->[11]。需要注意的是，squeeze只能对size=1的维度进行操作，如果对size不等于1的维度操作将不会有任何改变。

由于模型还没有经过预训练，所以目前模型输出的token都是随机的，没有任何含义。

**小结**：本文从GPT模型的结构说明开始，一步一步创建了模型推理过程中用到的每个组件，并演示了每个组件对输入数据运算的效果。随后基于这些组件封装出了我们自己的模型类MiniGPT，并基于这个模型类进行了自回归生成文本序列的演示，目前生成的序列是没有含义的，需要对这个模型进行训练后才能像GPT一样生成有含义的文本。